# Basic RAG with LlamaIndex (Gemini)
Query a local text file using semantic node splitting and a vector index.

In [1]:
import os
from pathlib import Path

import nest_asyncio
from dotenv import load_dotenv
from llama_index.core import Settings, VectorStoreIndex
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.readers.file import FlatReader

load_dotenv()
nest_asyncio.apply()  # lets the Gemini client run inside Jupyter
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise RuntimeError("GOOGLE_API_KEY not found. Add it to your .env file.")

In [2]:
# Configure LlamaIndex settings
Settings.llm = GoogleGenAI(model="gemini-2.5-flash-lite", max_tokens=100, api_key=GOOGLE_API_KEY)
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001", api_key=GOOGLE_API_KEY)

2026-10-13 07:27:41,610 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite "HTTP/1.1 200 OK"


In [3]:
# Semantic node splitter
splitter = SemanticSplitterNodeParser(
    buffer_size=1,
    breakpoint_percentile_threshold=95,
    embed_model=Settings.embed_model,
)

In [4]:
def parse_nodes(text_doc='sample_text.txt'):
    documents = FlatReader().load_data(Path(text_doc))
    return splitter.get_nodes_from_documents(documents)

In [5]:
# Vector index and query engine
nodes = parse_nodes()
index = VectorStoreIndex(nodes)
query_engine = index.as_query_engine(similarity_top_k=2)

2026-10-13 07:27:45,128 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


2026-10-13 07:27:46,003 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


In [6]:
def query_doc(user_query):
    if not user_query or not user_query.strip():
        return None
    return query_engine.query(user_query)

In [7]:
response = query_doc("What computer did the author buy around 1980?")
print("Query Response:")
print(response)

2026-10-13 07:27:46,749 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


2026-10-13 07:27:46,801 - INFO - AFC is enabled with max remote calls: 10.


2026-10-13 07:27:48,080 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


Query Response:
The author purchased an Apple II computer around 1980.
